In [38]:
from dotenv import load_dotenv
import os

load_dotenv()
print(f"API_KEY: {os.getenv('DASHSCOPE_API_KEY')}") 
print(f"API_URL: {os.getenv('DASHSCOPE_API_URL')}")
print(f"API_MODEL: {os.getenv('DEFAULT_LLM_MODEL')}")

API_KEY: sk-d04de33360574dd6bd8224dc1d9897a3
API_URL: https://dashscope.aliyuncs.com/compatible-mode/v1
API_MODEL: qwen3-32b


### 简单方式爬取非结构化信息
TODO：需要改写成真正的网页爬取逻辑

In [3]:
from langchain_community.document_loaders import WebBaseLoader
import os

os.environ['USER_AGENT'] = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
web_retriever_content = []

doc_white_list = [
       'https://fund.eastmoney.com', 
       'https://finance.sina.com.cn'
   ]

for url in doc_white_list:
    loader = WebBaseLoader(url)
    data = loader.load()
    web_retriever_content+=data


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# 去除空格
for doc in web_retriever_content:
    doc.page_content = " ".join(doc.page_content.split())

### 文本份块并封装成Langchain document对象 （含metadata信息）
TODO：如何加入有效的metadata信息

In [ ]:
# 切分文档
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=0, encoding='utf-8')
docs = text_splitter.split_documents(web_retriever_content)

In [ ]:
# 使用dashscope的embedding模型 + langchain的chroma库，构建本地向量数据库
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import Chroma
import os

embeddings = DashScopeEmbeddings(model="text-embedding-v3", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"))
vector_store = Chroma(persist_directory="chroma_db", collection_name="news", embedding_function=embeddings)
# vector_store.delete_collection()
vector_store.add_documents(docs)

vector_store.similarity_search("基金涨幅",k=3)


[Document(metadata={'description': '天天基金-东方财富旗下专业的基金交易平台。大数据多维度助你选出好基金，申购费率1折起，投资理财轻松上手。提供基金交易、金融资讯、收益查询等全方位服务。', 'title': '天天基金网(1234567.com.cn) --首批独立基金销售机构-- 东方财富网旗下基金平台!', 'source': 'https://fund.eastmoney.com', 'language': 'No language found.'}, page_content='10.47% 37.11% 9.32% 0.00% 购买 点击进入基金超市 序号 基金代码 基金简称 基金类型 单位净值 日期 日增长率 近1周 近1月 近3月 近6月 近1年 近3年 手续费 操作 1 021298 中欧北证50成份指数发起A 指数型-股票 1.9999 08-06 1.50% 1.30% 2.91% 5.80% 25.28% 127.91% -- 0.10% 购买 2 021299 中欧北证50成份指数发起C 指数型-股票 1.9937 08-06 1.50% 1.30% 2.89% 5.74% 25.11% 127.33% -- 0.00% 购买 3 018128 博时北证50成份指数发起式A 指数型-股票 1.6966 08-06 1.50% 1.30% 2.90% 6.54% 28.76% 124.15% -- 0.10% 购买 4 018129 博时北证50成份指数发起式C 指数型-股票 1.6795 08-06 1.50% 1.28% 2.86% 6.43% 28.51% 123.25% -- 0.00% 购买 5 017512 广发北证50成份指数A 指数型-股票'),
 Document(metadata={'description': '天天基金-东方财富旗下专业的基金交易平台。大数据多维度助你选出好基金，申购费率1折起，投资理财轻松上手。提供基金交易、金融资讯、收益查询等全方位服务。', 'source': 'https://fund.eastmoney.com', 'language': 'No language found.', 'title': '天天基金网(1234567.com.cn) --首批独立

### 使用比较原生的方式构建向量数据库
TODO：加入百炼的embedding模型

In [ ]:
# model: text-embedding-v3
# dimensions: 1024
# text-type: document

# setup local chroma DB

import chromadb

from chromadb.config import Settings

Settings.CHROMA_SETTINGS = {
    "chroma_db_impl": "duckdb+parquet",
    "persist_directory": "chroma_db",
}

client = chromadb.Client()
collection = client.get_or_create_collection("news")

In [23]:
import uuid

# /Users/dongwei/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:11<00:00, 7.27MiB/s]
collection.add(
    documents=[x.page_content for x in docs],
    metadatas=[x.metadata for x in docs],
    ids=[str(uuid.uuid4()) for _ in range(len(docs))]
)

In [27]:
results = collection.query(
    query_texts=["今日涨幅"],
    n_results=3,
    where={"source": "https://fund.eastmoney.com"}
)
results

{'ids': [['0cb7bb85-8ee2-4b73-a753-b4de40ce0e90',
   '4a71e673-a9e5-4ab9-8399-f57f0aa79700',
   '1bb5e2c4-1877-4f94-8eba-6f8bad9ae9c9']],
 'embeddings': None,
 'documents': [['债券型 指数型',
   '违法和不良信息举报:021-54509966/021-24099099',
   '16:28:08611医疗板块儿估值比较低长期来看机会大于风险相机里的猫男08-07 16:28:0732615牛市波段猎鹰08-07 16:28:0632244市场基动熵变08-07 16:28:0631405牛市基海墨痕深108-07 16:28:05173138这篇帖子适合所有普通投资者阅读。大家都想通过纳指十万点08-07 16:28:0332005创新价值墨痕08-07 16:27:5632594创新基海墨舞z08-07 16:27:5432276创新价值墨韵108-07 16:27:53161大盘要见顶了，大盘见顶之时，沙特启动之日。不要碰一字板08-07 16:27:4833065大涨痴心_绝对08-07 16:27:4231795创新钱波行08-07 16:27:39521医疗板块调整就是入手的机会相机里的猫男08-07 16:27:35点击查看更多基金吧热贴> 基金公司友情链接 A安联基金|专区 官网安信基金|专区 官网B博道基金|专区 官网渤海汇金|专区 官网北京京管泰富基金|专区 官网百嘉基金|专区 官网贝莱德基金管理|专区 官网博时基金|专区 官网北信瑞丰|专区 官网宝盈基金|专区']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'https://fund.eastmoney.com',
    'language': 'No language found.',
    'description': '天天基金-东方财富旗下专业的基金交易平台。大数据多维度助你选出好基金，申购费率1折起，投资理财轻松上手。提供